# Module A: object identifier (QuickDraw, 100 everyday objects)
Writes **`module_a.pt`** (+ `classes.json`) and exposes `predict(img) -> (labels, probs)`.

**Nothing repeats:** the dataset is downloaded once into `quickdraw100.npz`; the model is trained once and saved to `module_a.pt`. Re-running any cell, or restarting the kernel and running all, loads the saved model instead of training. To train again on purpose, set `RETRAIN = True` in the first cell (it flips itself back to `False` after training).

Colab: Runtime > Change runtime type > GPU first.

In [1]:
import os, json, time, urllib.parse, urllib.request, subprocess, sys
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageFilter

# ---------- settings ----------
CLASSES = ["airplane", "alarm clock", "apple", "backpack", "banana", "bandage", "baseball", "basket", "basketball",
 "bathtub", "bear", "bed", "bench", "bicycle", "birthday cake", "book", "bread", "broom", "bucket", "bus",
 "calculator", "calendar", "camera", "candle", "car", "carrot", "cell phone", "chair", "clock", "coffee cup",
 "computer", "cookie", "couch", "crayon", "cup", "door", "dresser", "envelope", "eraser", "eyeglasses",
 "fan", "flashlight", "fork", "frying pan", "guitar", "hammer", "hat", "headphones", "house", "ice cream",
 "jacket", "key", "keyboard", "knife", "ladder", "laptop", "light bulb", "lipstick", "map", "marker",
 "microphone", "microwave", "mug", "necklace", "oven", "paintbrush", "pants", "paper clip", "pencil", "piano",
 "pillow", "pizza", "pliers", "radio", "remote control", "saw", "scissors", "screwdriver", "shoe", "shorts",
 "shovel", "sink", "soccer ball", "sock", "spoon", "stairs", "stove", "suitcase", "sweater", "t-shirt",
 "table", "teapot", "telephone", "television", "toaster", "toothbrush", "umbrella", "vase", "wine bottle", "wristwatch"]
assert len(CLASSES) == len(set(CLASSES)) == 100
PER_CLASS, EPOCHS, BS, LR, SEED = 6000, 8, 512, 4e-3, 0     # LR = peak of the OneCycle schedule
URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy"
CACHE, CKPT = "quickdraw100.npz", "module_a.pt"
RETRAIN = False        # True = train again even though module_a.pt exists. It switches itself back to False after training.
dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED); print("device:", dev)

# ---- keep-in-memory helper: an object is built once per kernel; re-running a cell re-uses it ----
_mem = globals().setdefault("_mem", {})
def once(name, key, build, verbose=True):
    if name in globals() and _mem.get(name) == key:
        if verbose: print(f"[memory] {name}: already loaded, nothing recomputed")
        return globals()[name]
    globals()[name] = build(); _mem[name] = key; return globals()[name]


device: cpu


In [2]:
# ---------- data: download ONLY the first PER_CLASS drawings of each class, cache them in quickdraw100.npz ----------
def fetch(name, n):
    need = 256 + n * 784                                   # npy header (~128 B) + n rows of 784 bytes
    for attempt in range(5):
        try:
            req = urllib.request.Request(URL.format(urllib.parse.quote(name)), headers={"Range": f"bytes=0-{need - 1}"})
            with urllib.request.urlopen(req, timeout=60) as r: buf = r.read(need)
            assert buf[:6] == b"\x93NUMPY"
            off = 10 + int.from_bytes(buf[8:10], "little")  # where the pixel data starts
            x = np.frombuffer(buf[off:off + n * 784], np.uint8)
            assert x.size == n * 784
            return x.reshape(n, 784)
        except Exception as e:
            print(f"  {name}: retry {attempt + 1} ({e})"); time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"could not download {name}")

def load_data():
    if os.path.exists(CACHE):
        d = np.load(CACHE); print("data: loaded from", CACHE); return d["X"], d["y"]
    X, y = [], []
    for i, c in enumerate(CLASSES):
        print(f"[{i + 1}/100] {c}", end="\r"); X.append(fetch(c, PER_CLASS)); y.append(np.full(PER_CLASS, i))
    X, y = np.concatenate(X), np.concatenate(y); np.savez(CACHE, X=X, y=y); return X, y

def build_data():
    X, y = load_data(); X, y = torch.from_numpy(X).to(dev), torch.from_numpy(y).to(dev)    # whole dataset on the device
    p = torch.randperm(len(y), generator=torch.Generator().manual_seed(SEED)).to(dev)
    n5 = len(y) // 20                                       # 5% test, 5% val, 90% train (fixed split, same every run)
    return dict(X=X, y=y, te=p[:n5], va=p[n5:2 * n5], tr=p[2 * n5:])

def get_data(verbose=False): return once("DATA", (PER_CLASS, SEED), build_data, verbose)   # loaded once per kernel, only when needed

# ---------- model ----------
def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
class Net(nn.Module):                                       # 28 -> 14 -> 7 -> 3, global average pool, linear head
    def __init__(s, n):
        super().__init__()
        s.f = nn.Sequential(block(1, 32), block(32, 32), nn.MaxPool2d(2), block(32, 64), block(64, 64), nn.MaxPool2d(2),
                            block(64, 128), block(128, 128), nn.MaxPool2d(2), block(128, 256))
        s.head = nn.Sequential(nn.Dropout(.3), nn.Linear(256, n))
    def forward(s, x): return s.head(s.f(x).mean((2, 3)))

def inp(xb): return xb.float().div(255).view(-1, 1, 28, 28)      # uint8 (B,784) -> float (B,1,28,28)

@torch.no_grad()
def evaluate(net, X, y):
    net.eval(); loss = t1 = t5 = 0
    for i in range(0, len(y), 4096):
        lg, yb = net(inp(X[i:i + 4096])), y[i:i + 4096]
        loss += F.cross_entropy(lg, yb, reduction="sum").item()
        t1 += (lg.argmax(1) == yb).sum().item(); t5 += (lg.topk(5, 1).indices == yb[:, None]).any(1).sum().item()
    return loss / len(y), t1 / len(y), t5 / len(y)


In [3]:
# ---------- train (only if module_a.pt is missing or RETRAIN = True) or load, then freeze. Kept in memory afterwards. ----------
def build_net():
    net = Net(100).to(dev)
    if os.path.exists(CKPT) and not RETRAIN:
        net.load_state_dict(torch.load(CKPT, map_location=dev)["state_dict"]); print("loaded", CKPT, "- NO training")
    else:
        d = get_data(); X, y, tr, va, te = d["X"], d["y"], d["tr"], d["va"], d["te"]
        steps = len(tr) // BS
        opt = torch.optim.Adam(net.parameters(), LR)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=EPOCHS, steps_per_epoch=steps)
        amp = dev == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=amp)
        best = (1e9, None)
        for ep in range(EPOCHS):
            net.train(); t0 = time.time(); order = tr[torch.randperm(len(tr), device=dev)]; run = 0
            for s in range(steps):
                idx = order[s * BS:(s + 1) * BS]
                xb = torch.roll(inp(X[idx]), tuple(np.random.randint(-2, 3, 2).tolist()), (2, 3))   # small random shift
                with torch.autocast(dev, enabled=amp):
                    loss = F.cross_entropy(net(xb), y[idx], label_smoothing=.05)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                run += loss.item()
            vl, v1, v5 = evaluate(net, X[va], y[va])
            print(f"epoch {ep + 1}/{EPOCHS}  train {run / steps:.3f}  val {vl:.3f}  top1 {v1:.2%}  top5 {v5:.2%}  {time.time() - t0:.0f}s")
            if vl < best[0]: best = (vl, {k: v.clone() for k, v in net.state_dict().items()})   # best on validation only
        net.load_state_dict(best[1])
        _, t1, t5 = evaluate(net, X[te], y[te]); print(f"\nTEST  top-1 {t1:.2%}  top-5 {t5:.2%}")
        torch.save({"state_dict": net.state_dict(), "classes": CLASSES}, CKPT); json.dump(CLASSES, open("classes.json", "w")); print("saved", CKPT)
    net.eval()
    for q in net.parameters(): q.requires_grad_(False)      # frozen from here on
    return net

if RETRAIN: _mem.pop("net", None)                            # force a rebuild only when retraining was asked for
net = once("net", (CKPT,), build_net)
RETRAIN = False                                              # so the next run of this cell never retrains by accident


loaded module_a.pt - NO training


In [4]:
# ---------- input handling and the predict() API used by the rest of the pipeline ----------
def prep(img):
    """RGB/RGBA/gray array or PIL image (drawing, phone photo, upload) -> (28,28) float in [0,1], or None if blank."""
    if isinstance(img, dict): img = img["composite"]
    im = Image.fromarray(img) if isinstance(img, np.ndarray) else img
    if im.mode == "RGBA": im = Image.alpha_composite(Image.new("RGBA", im.size, "white"), im)
    g = np.asarray(im.convert("L"), np.float32)
    dev_ = np.abs(g - np.median(g))                         # distance from paper colour: works for dark-on-light and light-on-dark
    mx = dev_.max()
    if mx < 20: return None                                 # nothing drawn
    ink = np.where(dev_ > .3 * mx, dev_ / mx, 0)
    ys, xs = np.where(ink > 0)
    ink = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    s = max(ink.shape); sq = np.zeros((s, s), np.float32)   # pad to a square, drawing centred (QuickDraw style)
    sq[(s - ink.shape[0]) // 2:(s - ink.shape[0]) // 2 + ink.shape[0], (s - ink.shape[1]) // 2:(s - ink.shape[1]) // 2 + ink.shape[1]] = ink
    t = Image.fromarray((sq * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(2 * (s // 56) + 1))   # thicken so strokes survive shrinking
    a = np.asarray(t.resize((28, 28), Image.BOX), np.float32)
    return a / max(a.max(), 1)

@torch.no_grad()
def predict(img):
    """img: (H,W), (1,H,W) or (B,1,H,W); [0,1] or 0-255; WHITE strokes on BLACK; any size (e.g. 256x256 from the pipeline).
    returns (labels: list[str], probs: (B,100) tensor)."""
    x = torch.as_tensor(np.asarray(img), dtype=torch.float32)
    if x.max() > 1.5: x = x / 255
    x = x.reshape(-1, 1, x.shape[-2], x.shape[-1]).to(dev)
    if x.shape[-2:] != (28, 28): x = F.adaptive_avg_pool2d(x, 28)
    pr = net(x).softmax(1).cpu()
    return [CLASSES[i] for i in pr.argmax(1)], pr

def predict_file(path, k=5):
    """Classify a drawing/photo file (png/jpg): prints and returns the top-k (label, probability)."""
    a = prep(Image.open(path))
    if a is None: print("no drawing found"); return []
    top = predict(a)[1][0].topk(k); out = [(CLASSES[i], float(p)) for p, i in zip(top.values, top.indices)]
    for l, p in out: print(f"  {l:15s} {p:.1%}")
    return out
# usage: predict_file("my_cup.png")


In [5]:
# ---------- evaluate on the held-out test split: accuracy, calibration, most confused classes ----------
d = get_data(); te = d["te"]; Xte, yt = d["X"][te], d["y"][te]
with torch.no_grad():
    pr = torch.cat([net(inp(Xte[i:i + 4096])).softmax(1) for i in range(0, len(te), 4096)])
conf, pred = pr.max(1); correct = (pred == yt).float()
print(f"TEST ({len(te)} drawings never used for training or model selection)")
print(f"  top-1 {correct.mean().item():.2%} | top-5 {(pr.topk(5, 1).indices == yt[:, None]).any(1).float().mean().item():.2%}")
bins = torch.linspace(0, 1, 11, device=pr.device); ece = 0.
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.any(): ece += m.float().mean().item() * abs(correct[m].mean().item() - conf[m].mean().item())
print(f"  calibration: expected calibration error {ece:.3f} | mean confidence {conf.mean().item():.2f} vs accuracy {correct.mean().item():.2f}")
pc = torch.stack([correct[yt == c].mean() for c in range(100)]).cpu().numpy()
print("  hardest classes:", ", ".join(f"{CLASSES[i]} {pc[i]:.0%}" for i in np.argsort(pc)[:6]))
cm = torch.zeros(100, 100, dtype=torch.long); cm.index_put_((yt.cpu(), pred.cpu()), torch.ones(len(yt), dtype=torch.long), accumulate=True); cm.fill_diagonal_(0)
top = cm.flatten().topk(6); print("  most frequent confusions (true -> predicted):", ", ".join(f"{CLASSES[int(i) // 100]}->{CLASSES[int(i) % 100]} x{int(v)}" for v, i in zip(top.values, top.indices)))


data: loaded from quickdraw100.npz
TEST (30000 drawings never used for training or model selection)
  top-1 82.67% | top-5 95.41%
  calibration: expected calibration error 0.057 | mean confidence 0.77 vs accuracy 0.83
  hardest classes: marker 48%, cup 49%, oven 51%, crayon 55%, coffee cup 55%, telephone 59%
  most frequent confusions (true -> predicted): coffee cup->mug x86, telephone->cell phone x71, cup->mug x71, computer->laptop x66, sweater->t-shirt x47, jacket->sweater x46


In [6]:
# ---------- unit tests (plan step A3): output shape, determinism, frozen weights ----------
x = np.random.default_rng(0).random((3, 256, 256)).astype(np.float32)
l1, p1 = predict(x); l2, p2 = predict(x)
assert p1.shape == (3, 100) and len(l1) == 3, "wrong output shape"
assert torch.allclose(p1.sum(1), torch.ones(3), atol=1e-4), "probabilities must sum to 1"
assert torch.equal(p1, p2) and l1 == l2, "predict() must be deterministic"
assert not any(q.requires_grad for q in net.parameters()) and not net.training, "weights must be frozen / eval mode"
print("all Module A tests passed")


all Module A tests passed


### Optional app
Run only if you want the draw / camera / upload interface.

In [7]:
# ---------- OPTIONAL app: draw / camera / upload. Run this cell only when you want the app. ----------
LAUNCH_APP = True
if LAUNCH_APP:
    for k in ("NO_PROXY", "no_proxy"): os.environ[k] = "localhost,127.0.0.1,0.0.0.0"   # stops a proxy/VPN from blocking Gradio's local check
    try: import gradio as gr
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"]); import gradio as gr
    def classify(img):
        if img is None: return {}
        a = prep(img)
        if a is None: return {"(nothing drawn yet)": 1.0}
        top = predict(a)[1][0].topk(5)
        return {CLASSES[i]: float(p) for p, i in zip(top.values, top.indices)}
    app = gr.TabbedInterface(
        [gr.Interface(classify, gr.Sketchpad(), gr.Label(num_top_classes=5), live=True),
         gr.Interface(classify, gr.Image(sources=["webcam", "upload"]), gr.Label(num_top_classes=5))], ["Draw", "Camera / Upload"])
    gr.close_all()                                          # shuts down servers left over from earlier runs in this kernel
    if "google.colab" in sys.modules: app.launch(share=True)                 # Colab: public link (needed for the webcam)
    else: app.launch(server_name="127.0.0.1", inbrowser=True)               # local: webcam works on localhost


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Using existing dataset file at: .gradio\flagged\dataset1.csv


Traceback (most recent call last):
  File "c:\Users\welcome\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\welcome\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\welcome\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\welcome\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignor